<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Bonus Demo - Building Single Agents with LangChain
## Overview
In this demo, you will build a single Data Analyst agent using LangChain on Databricks. You will define a custom tool with LangChain's `@tool` decorator, register it to Unity Catalog, discover it via a Managed MCP server, and connect it to a LangChain agent with MLflow tracing.

## Learning Objectives
By the end of this demo, you will be able to:
1. Define custom tools using LangChain's `@tool` decorator
2. Build and test a LangChain agent with LangGraph's `create_react_agent`
3. Register tools to Unity Catalog for governance
4. Discover UC tools via Managed MCP servers
5. Rebuild the agent with MCP-sourced tools
6. Enable MLflow autologging and inspect execution traces


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
This notebook is meant to mimic notebook <strong>05 Demo - Building Single Agents with the OpenAI Agents SDK</strong>.
</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-9

### A1. Import Libraries

We import the core components needed for this demo:
- `ChatDatabricks` for connecting to Databricks model serving endpoints
- `@tool` decorator for defining agent tools
- `create_react_agent` from LangGraph for building the agent

In [0]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from databricks_langchain import ChatDatabricks

print("LangChain + LangGraph components imported.")

## B. Define the Data Analyst Agent

### B1. Define the `classify_price_tier` Tool

The `classify_price_tier` tool classifies an Airbnb listing's nightly price into one of four market tiers based on the SF pricing distribution:

| Tier | Price Range | Percentile Band |
|---|---|---|
| **Budget** | Under $100 | Bottom 25% |
| **Mid-Range** | $100 -- $174 | 25--50% |
| **Premium** | $175 -- $299 | 50--75% |
| **Luxury** | $300+ | Top 25% |

LangChain's `@tool` decorator registers this function as a tool that the agent can invoke. The SDK extracts the function signature and docstring to generate the tool schema that the LLM uses to decide when and how to call the tool.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
An example production workflow is: prototype with <code>@tool</code>, then register to Unity Catalog, then access via a Managed MCP server. This demo follows that full progression. In Section C, we register the function to Unity Catalog. In Section D, we connect via MCP.
</p>
</div>
</div>
</div>

In [0]:
import json

@tool
def classify_price_tier(price: float) -> str:
    """Classify an Airbnb listing price into a market tier based on SF pricing distribution.

    Uses hardcoded SF Airbnb market thresholds to label a nightly price as
    Budget, Mid-Range, Premium, or Luxury, and returns the corresponding
    percentile band for context.

    Useful for questions like: 'Is $180/night a good deal in SF?',
    'What tier does this listing fall into?'

    Returns a JSON object with tier, percentile_band, and a short interpretation.

    Args:
        price: Nightly listing price in USD.
    """
    tiers = [
        (100,  "Budget",    "bottom 25%",  "Well below the SF market average."),
        (175,  "Mid-Range", "25-50%",      "Around the SF market median."),
        (300,  "Premium",   "50-75%",      "Above average for SF listings."),
        (float("inf"), "Luxury", "top 25%", "Among the most expensive listings in SF."),
    ]

    if price < 0:
        return json.dumps({"error": "Price must be a positive value."})

    for threshold, tier, band, interpretation in tiers:
        if price < threshold:
            return json.dumps({
                "price": price,
                "tier": tier,
                "percentile_band": band,
                "interpretation": interpretation
            }, indent=2)

print(f"Defined tool: {classify_price_tier.name}")

### B2. Create the Agent

With the tool defined, we create a LangGraph agent using `create_react_agent`. This takes three inputs:
1. **LLM**: `ChatDatabricks` connects to a Databricks model serving endpoint
2. **Tools**: The list of tools the agent can call
3. **Prompt**: A system prompt that shapes the agent's behavior

The agent runs a ReAct loop: the LLM reasons about which tool to call, executes it, observes the result, and generates a final response.

In [0]:
llm = ChatDatabricks(
    endpoint="databricks-gpt-5-2",
    temperature=0.1,
)

system_prompt = "You are a data analyst that helps users understand SF Airbnb listing prices. Use the classify_price_tier tool to categorize prices into market tiers. Provide clear, concise answers."

agent_executor = create_react_agent(llm, [classify_price_tier], prompt=system_prompt)

print("Agent created with inline @tool.")

### B3. Test the Agent

We send three progressively complex queries to validate that the agent correctly invokes the tool and interprets the results.

1. A straightforward pricing question that requires a single tool call.

In [0]:
query_1 = "Is $150/night a good deal in SF?"
result_1 = agent_executor.invoke({"messages": [("human", query_1)]})
print(f"User:     {query_1}")
print(f"Response: {result_1['messages'][-1].content}")

2. A direct tier classification at the high end of the price spectrum.

In [0]:
query_2 = "What tier would a $500/night luxury listing fall into?"
result_2 = agent_executor.invoke({"messages": [("human", query_2)]})
print(f"User:     {query_2}")
print(f"Response: {result_2['messages'][-1].content}")

3. Multiple prices in a single request, testing whether the agent calls the tool multiple times.

In [0]:
query_3 = "Classify these prices: $80, $200, $350"
result_3 = agent_executor.invoke({"messages": [("human", query_3)]})
print(f"User:     {query_3}")
print(f"Response: {result_3['messages'][-1].content}")

## C. Register Tools to Unity Catalog

### C1. Register `classify_price_tier` as a Python UC Function

The inline `@tool` is local to this notebook session. To make the tool governed, discoverable, and reusable across agents and workspaces, we register it to Unity Catalog as a Python function using `DatabricksFunctionClient`.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The function registered below is a <strong>plain Python function</strong> (no <code>@tool</code> decorator). The <code>DatabricksFunctionClient</code> inspects the function signature and docstring to create the UC function metadata. The logic is identical to the tool we prototyped above.
</p>
</div>
</div>
</div>

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

def classify_price_tier_uc(price: float) -> str:
    """Classify an Airbnb listing price into a market tier based on SF pricing distribution.

    Uses hardcoded SF Airbnb market thresholds to label a nightly price as
    Budget, Mid-Range, Premium, or Luxury, and returns the corresponding
    percentile band for context.

    Useful for questions like: 'Is $180/night a good deal in SF?',
    'What tier does this listing fall into?'

    Returns a JSON object with tier, percentile_band, and a short interpretation.

    Args:
        price: Nightly listing price in USD.
    """
    import json

    tiers = [
        (100,  "Budget",    "bottom 25%",  "Well below the SF market average."),
        (175,  "Mid-Range", "25-50%",      "Around the SF market median."),
        (300,  "Premium",   "50-75%",      "Above average for SF listings."),
        (float("inf"), "Luxury", "top 25%", "Among the most expensive listings in SF."),
    ]

    if price < 0:
        return json.dumps({"error": "Price must be a positive value."})

    for threshold, tier, band, interpretation in tiers:
        if price < threshold:
            return json.dumps({
                "price": price,
                "tier": tier,
                "percentile_band": band,
                "interpretation": interpretation
            }, indent=2)

client = DatabricksFunctionClient()
function_info = client.create_python_function(
    func=classify_price_tier_uc,
    catalog=catalog_name,
    schema=schema_name,
    replace=True,
)

print(f"Created UC function: {function_info.full_name}")

### C2. Verify Registered Functions

Confirm that the function is visible in the Unity Catalog schema, then call it directly to verify it returns the expected output.
1. Navigate to the **Catalog Explorer** on the left side menu and search for `classify_price_tier_uc`
1. Run the next cell to test the output from calling the function from UC

In [0]:
result = spark.sql(f"SELECT {catalog_name}.{schema_name}.classify_price_tier_uc(150.0) AS result")
display(result)

## D. Discover Tools via MCP

### D1. Connect to the Managed MCP Server

Databricks provides **Managed MCP servers** that automatically expose Unity Catalog functions as tools. Any agent can discover and call these functions through the MCP protocol without custom integration code.

The MCP server endpoint for UC functions follows this pattern:
`https://workspace/api/2.0/mcp/functions/catalog/schema`

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
We use <code>DatabricksMCPClient</code> from <code>databricks_mcp</code> to interact with the MCP endpoint in a notebook context.
See: Managed MCP servers (<a href="https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">GCP</a>)
</p>
</div>
</div>
</div>

In [0]:
import nest_asyncio
nest_asyncio.apply()

from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
workspace_host = ws.config.host

# Point to the MCP server for our catalog/schema
mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

# Discover available tools
tools_discovered = mcp_client.list_tools()
print(f"Discovered {len(tools_discovered)} UC function tool(s) via MCP:")
for t in tools_discovered:
    print(f"  - {t.name}: {t.description}")

### D2. Call the UC Function Through MCP

Now we call the registered UC function through the MCP protocol, just as an agent would. The default tool name is of the form `catalogName__schemaName__toolName`.

In [0]:
tool_name = f"{catalog_name}__{schema_name}__classify_price_tier_uc"
result = mcp_client.call_tool(tool_name, {"price": 120.00})
print(f"Result: {''.join([c.text for c in result.content])}")

### D3. Load MCP Tools into LangChain

We convert the MCP tools we discovered into LangChain-compatible `StructuredTool` objects. Each tool wraps a call to `mcp_client.call_tool()`, so tool invocations route through MCP at runtime.

<div style="display:flex;align-items:center;gap:0;margin:20px 0;flex-wrap:wrap;font-family:system-ui,sans-serif;">
  <div style="background:#F9F7F4;border:2px solid #1B5162;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">@tool<br/><span style="font-weight:400;font-size:12px;">Local only</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#2574B5;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">UC Function<br/><span style="font-weight:400;font-size:12px;">Governed + reusable</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#02A36F;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">MCP Server<br/><span style="font-weight:400;font-size:12px;">Agent-accessible + standardized</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#FF5F46;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">LangChain Agent<br/><span style="font-weight:400;font-size:12px;">Production-ready</span></div>
</div>

This is the recommended progression: start with `@tool` for rapid prototyping, register to UC for governance, then expose through MCP for production agent access.

In [0]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

def _make_mcp_tool(mcp_tool_meta):
    """Wrap a single MCP tool as a LangChain StructuredTool."""
    tool_name = mcp_tool_meta.name

    # Build a Pydantic model from the MCP tool's input schema
    schema_props = mcp_tool_meta.inputSchema.get("properties", {})
    field_definitions = {}
    for prop_name, prop_info in schema_props.items():
        prop_type = {"string": str, "number": float, "integer": int, "boolean": bool}.get(
            prop_info.get("type", "string"), str
        )
        field_definitions[prop_name] = (
            prop_type,
            Field(description=prop_info.get("description", "")),
        )
    args_model = type(f"{tool_name}_args", (BaseModel,), {"__annotations__": {k: v[0] for k, v in field_definitions.items()}, **{k: v[1] for k, v in field_definitions.items()}})

    def _invoke(**kwargs):
        result = mcp_client.call_tool(tool_name, kwargs)
        return "".join([c.text for c in result.content])

    return StructuredTool.from_function(
        func=_invoke,
        name=tool_name,
        description=mcp_tool_meta.description or "",
        args_schema=args_model,
    )

mcp_langchain_tools = [_make_mcp_tool(t) for t in tools_discovered]

print(f"Loaded {len(mcp_langchain_tools)} MCP tool(s) as LangChain tools:")
for t in mcp_langchain_tools:
    print(f"  - {t.name}")

## E. Enable MLflow Tracing and Run the MCP-Powered Agent

### E1. Enable Autologging

MLflow autologging for LangChain automatically captures execution traces for every agent run. Each trace records the full sequence of LLM calls, tool invocations, inputs, outputs, and latencies without requiring any code changes to the agent itself.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
When using LangChain, use <code>mlflow.langchain.autolog()</code> (compared to <code>mlflow.openai.autolog()</code> in the OpenAI Agents SDK demos). Autologging captures: LLM call spans (model, prompt, completion tokens), tool call spans (function name, arguments, return value), and the overall agent run span with total latency.
</p>
</div>
</div>
</div>

In [0]:
import mlflow

mlflow.langchain.autolog()

print("MLflow LangChain autologging enabled.")
print(f"Traces will appear in experiment: {mlflow_experiment_path}")

### E2. Build the MCP-Powered Agent

We rebuild the agent using the same LLM and system prompt, but now the tools come from MCP rather than inline `@tool` definitions. The `create_react_agent` API is identical -- only the tool source changes.

In [0]:
mcp_agent_executor = create_react_agent(llm, mcp_langchain_tools, prompt=system_prompt)

print("Agent created with MCP-sourced tools.")
print(f"  Tools: {[t.name for t in mcp_langchain_tools]}")

### E3. Test the MCP-Powered Agent

With autologging enabled, each agent run produces a full execution trace. The trace captures the LLM reasoning step, the tool call to `classify_price_tier_uc` (now routed through MCP), and the final response generation.

In [0]:
traced_query = "I found a listing for $225/night in the Mission district. Is that a fair price?"
result = mcp_agent_executor.invoke({"messages": [("human", traced_query)]})
print(f"User:     {traced_query}")
print(f"Response: {result['messages'][-1].content}")

### E4. Run Additional Queries

In [0]:
result = mcp_agent_executor.invoke({"messages": [("human", query_1)]})
print(f"User:     {query_1}")
print(f"Response: {result['messages'][-1].content}")

In [0]:
result = mcp_agent_executor.invoke({"messages": [("human", query_3)]})
print(f"User:     {query_3}")
print(f"Response: {result['messages'][-1].content}")

## F. Conclusion

In this demo, you:

1. Defined a `classify_price_tier` tool using LangChain's `@tool` decorator to classify Airbnb prices into market tiers
2. Built a single Data Analyst agent with LangGraph's `create_react_agent` and tested it with progressively complex pricing queries
3. Registered `classify_price_tier` as a Python UC function for governance
4. Discovered the UC function through a Managed MCP server and wrapped it as a LangChain `StructuredTool`
5. Rebuilt the agent with MCP-sourced tools and enabled MLflow tracing with `mlflow.langchain.autolog()`

The same `@tool` to UC to MCP progression works across frameworks -- whether you use LangChain (this demo) or the OpenAI Agents SDK (notebook 05).

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>